Project Name: EECS 690 Mini-Project 06

Purpose: Regression analysis of marathon data using KNN and linear regression

Input(s): marathon.csv

Output(s): Multiple scatter plots, some with regression lines

Author: Jacob Kice

Initial Date: 04 / 09 / 2026

Updated Date: 04 / 09 / 2026

Part 1: KNN Regression Analysis

In [1]:
# Import libraries
import altair as alt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [2]:
set_config(transform_output="pandas")

In [3]:
# Read the marathon data
marathon = pd.read_csv('data/marathon.csv')
marathon

,age,bmi,female,footwear,group,injury,mf_d,mf_di,mf_ti,max,sprint,mf_s,time_hrs
0,35,23.592323,0,2,1,2,42195,4,10295,60.0,1,4.098592,2.859722
1,33,22.518295,0,2,2,2,42195,3,12292,50.0,0,3.432720,3.414444
2,38,25.560312,0,2,3,1,42195,4,10980,65.0,0,3.842896,3.050000
3,34,22.607931,0,2,1,1,42195,3,10694,88.0,1,3.945670,2.970556
4,39,24.974836,0,2,1,1,42195,2,13452,51.0,0,3.136708,3.736667
...,...,...,...,...,...,...,...,...,...,...,...,...,...
924,23,23.277760,1,2,2,1,42195,3,15660,18.0,0,2.694444,4.350000
925,30,24.489796,0,2,2,1,42195,2,16110,45.0,0,2.619181,4.475000
926,44,24.237617,0,2,3,1,42195,2,12289,63.0,1,3.433558,3.413611
927,34,21.249750,0,2,3,1,42195,3,12602,32.0,0,3.348278,3.500556


In [4]:
# Take a random subset of 50
marathon_50 = marathon.sample(n=50, random_state=42)
# Scatterplot
answer2 = alt.Chart(marathon_50).mark_circle(size=60, opacity=0.7).encode(
    x=alt.X('max', title='Max Distance Ran per Week During Training (miles)'),
    y=alt.Y('time_hrs', title='Race Time (hours)').scale(zero=False),
)
answer2

alt.Chart(...)

In [5]:
# Find the 4 nearest neighbors to 100 miles/week
new_point = 100
marathon_50['dist_to_100'] = np.abs(marathon_50['max'] - new_point)
nearest4 = marathon_50.nsmallest(4, 'dist_to_100')
answer3 = nearest4['time_hrs'].mean()
print(answer3)

2.699236111111111


In [6]:
# Prepare data
marathon_training, marathon_testing = train_test_split(marathon, train_size=0.75, random_state=42)
X_train = marathon_training[['max']]
y_train = marathon_training['time_hrs']
X_test = marathon_testing[['max']]
y_test = marathon_testing['time_hrs']
# Pipeline
marathon_pipe = make_pipeline(StandardScaler(), KNeighborsRegressor())
# Cross-validation
marathon_cv = cross_validate(
    marathon_pipe, X_train, y_train,
    cv=5,
    scoring='neg_root_mean_squared_error',
        
)
marathon_cv = pd.DataFrame(marathon_cv)
marathon_cv

,fit_time,score_time,test_score
0,0.003116,0.002289,-0.547594
1,0.002398,0.002005,-0.573537
2,0.002205,0.001706,-0.718484
3,0.002342,0.001652,-0.643909
4,0.002277,0.001738,-0.678315


In [7]:
# Parameter grid
param_grid = {'kneighborsregressor__n_neighbors': range(1, 201)}
marathon_tuned = GridSearchCV(
    marathon_pipe,
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=1
)
marathon_tuned.fit(X_train, y_train)
marathon_results = pd.DataFrame(marathon_tuned.cv_results_)
marathon_results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_kneighborsregressor__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.004748,0.001308,0.003923,0.001887,1,{'kneighborsregressor__n_neighbors': 1},-0.660110,-0.836843,-0.772439,-0.773979,-0.823174,-0.773309,0.062191,200
1,0.003863,0.001719,0.002294,0.000698,2,{'kneighborsregressor__n_neighbors': 2},-0.560673,-0.688064,-0.727112,-0.713300,-0.764580,-0.690746,0.069583,199
2,0.002248,0.000078,0.001732,0.000238,3,{'kneighborsregressor__n_neighbors': 3},-0.543664,-0.627253,-0.741342,-0.696217,-0.706265,-0.662948,0.070188,198
3,0.002206,0.000141,0.001576,0.000028,4,{'kneighborsregressor__n_neighbors': 4},-0.510354,-0.629260,-0.732213,-0.668440,-0.703034,-0.648660,0.077245,197
4,0.002290,0.000237,0.001666,0.000147,5,{'kneighborsregressor__n_neighbors': 5},-0.547594,-0.573537,-0.718484,-0.643909,-0.678315,-0.632368,0.063731,196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,0.002377,0.000081,0.003259,0.000138,196,{'kneighborsregressor__n_neighbors': 196},-0.530495,-0.561447,-0.647325,-0.637176,-0.627192,-0.600727,0.046208,167
196,0.002612,0.000259,0.003316,0.000190,197,{'kneighborsregressor__n_neighbors': 197},-0.531024,-0.561681,-0.647517,-0.637438,-0.628135,-0.601159,0.046196,170
197,0.002452,0.000193,0.003396,0.000248,198,{'kneighborsregressor__n_neighbors': 198},-0.531795,-0.562277,-0.647562,-0.637088,-0.627941,-0.601333,0.045792,171
198,0.002565,0.000270,0.003221,0.000129,199,{'kneighborsregressor__n_neighbors': 199},-0.532113,-0.563343,-0.647859,-0.636550,-0.628003,-0.601574,0.045499,172


In [8]:
marathon_min = marathon_tuned.best_params_
marathon_best_RMSPE = -marathon_tuned.best_score_
marathon_min

{'kneighborsregressor__n_neighbors': 85}

In [9]:

print(marathon_best_RMSPE)

0.5905580584184031


In [10]:
marathon_prediction = marathon_tuned.predict(X_test)
marathon_summary = mean_squared_error(y_test, marathon_prediction)
marathon_summary = np.sqrt(marathon_summary)
print(marathon_summary)

0.5623081416025674


In [11]:
# Add predictions to training data
marathon_preds = marathon_training.assign(predictions=marathon_tuned.predict(marathon_training[['max']]))
# Scatterplot with prediction line
marathon_plot = alt.Chart(marathon_preds).mark_circle(opacity=0.4).encode(
    x=alt.X('max', title='Max Distance Ran per Week During Training (miles)'),
    y=alt.Y('time_hrs', title='Race Time (hours)').scale(zero=False)
) + alt.Chart(marathon_preds).mark_line(color='black').encode(
    x='max',
    y='predictions'
)
marathon_plot

alt.LayerChart(...)

Part 2: Simple Linear Regression

In [12]:
# Import LinearRegression
from sklearn.linear_model import LinearRegression

In [13]:
# Scatterplot
marathon_scatter = alt.Chart(marathon_training).mark_point(opacity=0.4).encode(
    x=alt.X('max', title='Max Distance Ran per Week During Training (miles)'),
    y=alt.Y('time_hrs', title='Race Time (hours)').scale(zero=False)
)
marathon_scatter

alt.Chart(...)

In [14]:
lm = LinearRegression()
lm

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [15]:
lm_fit = lm.fit(X_train, y_train)
lm_fit

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [16]:
marathon_preds = marathon_training.assign(predictions=lm.predict(marathon_training[['max']]))
marathon_plot = alt.Chart(marathon_preds).mark_circle(opacity=0.4).encode(
    x=alt.X('max', title='Max Distance Ran per Week During Training (miles)'),
    y=alt.Y('time_hrs', title='Race Time (hours)').scale(zero=False)
) + alt.Chart(marathon_preds).mark_line(color='black').encode(
    x='max',
    y='predictions'
)
marathon_plot

alt.LayerChart(...)

In [17]:
marathon_preds_test = marathon_testing.assign(predictions=lm.predict(marathon_testing[['max']]))
marathon_plot_test = alt.Chart(marathon_preds_test).mark_circle(opacity=0.4).encode(
    x=alt.X('max', title='Max Distance Ran per Week During Training (miles)'),
    y=alt.Y('time_hrs', title='Race Time (hours)').scale(zero=False)
) + alt.Chart(marathon_preds_test).mark_line(color='black').encode(
    x='max',
    y='predictions'
)
marathon_plot_test

alt.LayerChart(...)

Part 3: Final Reflection

1. When might KNN regression be more appropriate than simple linear regression, and vice versa?

KNN regression is more apropriate relationship is non-linear, as KNN is capable of learning more complex relationships. On the other hand, simple linear regression is more appropriate when the relationship is more linear, as it is more efficient to learn and use and is easier to understand.

2. How does KNN regression differ from KNN classification, and what challenges arise when predicting continuous (vs. categorical) targets?

KNN regression predicts a continuous value while KNN classification predicts a discrete class or category. KNN regression works by taking the average of the values of the nearest neighbors, while KNN classification takes the most common class amongst the nearest neighbors. Predicting a continuous value can be more sensitive to outliers as they can skew the average, but they have less impact on class prediction as it is based on the most common neighbor class and thus is not skewed by outliers. Additionally, KNN regression, being based on the average of the nearest neighbors, is not capable of predicting values outside the bounds of the training data. KNN classification predicts from a set of categories, so this limitation does not apply.

3. Why is the linearity assumption in simple linear regression important, and what happens if it’s violated?

A simple linear regression model can only capture a linear relationship between the independent and dependent variables, so it assumes that the relationship is linear in order to calculate the necessary metrics such as slope and intercept. If the assumptoion is violated, the predictions of the model and the inferences drawn from them may be inaccurate or misleading.